# BE m228 Final Project

### Load Libraries & Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc, roc_curve
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.utils import resample
from sklearn.metrics import recall_score, precision_score, confusion_matrix

from ucimlrepo import fetch_ucirepo 



In [ ]:
diabetes_data = fetch_ucirepo(id=891)

X = diabetes_data.data.features 
X = X.drop('GenHlth', axis=1)
y = diabetes_data.data.targets

df = pd.DataFrame(X, columns=diabetes_data.variables['name'])
df['Diabetes_binary'] = y
df['ID'] = diabetes_data.data.ids
df.head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(class_weight="balanced", random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "max_features": ["sqrt"]
}

# this optimization takes ~4 minutes to run, so commented out for now; best params are n_estimators=200, max_depth=10, max_features="sqrt"
# grid = GridSearchCV(rf, param_grid, cv=3, scoring="f1", n_jobs=-1)
# grid.fit(X_train, y_train)

# print(grid.best_params_)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced", max_features="sqrt")
rf_model.fit(X_train, y_train)
rf_y_preds = rf_model.predict(X_test)

print(classification_report(y_test, rf_y_preds))

In [ ]:
rf_y_train_probs = rf_model.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, rf_y_train_probs)

# calculate F1 for every threshold
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Optimal Threshold found: {best_threshold:.4f}")

In [ ]:
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]

# apply the custom threshold
rf_test_preds = (rf_test_probs >= best_threshold).astype(int)

#### Model Metrics

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, rf_test_preds).ravel()

precision = tp / (tp + fp)
recall = tp / (tp + fn) # sensitivity
specificity = tn / (tn + fp)
accuracy = (tp + tn) / (tp + fp + tn + fn)

print(f'precision of model: {precision}')
print(f'recall of model: {recall}')
print(f'specificity of model: {specificity}')
print(f'accuracy of model: {accuracy}')

In [ ]:
pr_auc = average_precision_score(y_test, rf_test_probs)
brier_score = brier_score_loss(y_test, rf_test_probs)
roc_auc = roc_auc_score(y_test, rf_test_probs)
f1 = f1_score(y_test, rf_test_preds)

print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"Brier Score (Calibration): {brier_score:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"F1-Score: {f1:.4f}")

## XGBoost

Finding best parameters

In [ ]:
from xgboost import XGBClassifier

ratio = (y_train.values == 0).sum() / (y_train.values == 1).sum()

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200]
}

xgb_model = XGBClassifier(
    scale_pos_weight=ratio, 
    eval_metric='logloss',
    random_state=42
)

# optimization commented out to save time; best params are max_depth=3, learning_rate=0.1, n_estimators=100
# grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='f1', cv=3, verbose=1)
# grid_search.fit(X_train, y_train)

# print(f"Best Parameters: {grid_search.best_params_}")
# print(f"Best F1-Score from GridSearch: {grid_search.best_score_:.4f}")

Training, fitting, and predicting:

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

ratio = (y_train.values == 0).sum() / (y_train.values == 1).sum()

# initialize the Stratified K-Fold object
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# initialize the model 
# add scale_pos_weight to handle the imbalance during training
xg_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate = 0.1, scale_pos_weight=ratio, eval_metric='logloss', random_state=42)

# run cross-validation
# use 'f1' or 'roc_auc' because 'accuracy' is misleading for imbalanced data
cv_results = cross_validate(xg_model, X_train, y_train.values.ravel(), cv=skf, scoring=['roc_auc', 'f1', 'average_precision'])

In [ ]:
xg_model.fit(X_train, y_train.values.ravel())

# find the best threshold using the training data
xg_y_train_probs = xg_model.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, xg_y_train_probs)

# calculate F1 for every threshold
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Optimal Threshold found: {best_threshold:.4f}")

In [ ]:
xg_test_probs = xg_model.predict_proba(X_test)[:, 1]

# apply the custom threshold
xg_test_preds = (xg_test_probs >= best_threshold).astype(int)

#### Model Metrics

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, xg_test_preds).ravel()

precision = tp / (tp + fp)
recall = tp / (tp + fn) # sensitivity
specificity = tn / (tn + fp)
accuracy = (tp + tn) / (tp + fp + tn + fn)

print(f'precision of model: {precision}')
print(f'recall of model: {recall}')
print(f'specificity of model: {specificity}')
print(f'accuracy of model: {accuracy}')

In [ ]:
pr_auc = average_precision_score(y_test, xg_test_probs)
brier_score = brier_score_loss(y_test, xg_test_probs)
roc_auc = roc_auc_score(y_test, xg_test_probs)
f1 = f1_score(y_test, xg_test_preds)

print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"Brier Score (Calibration): {brier_score:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"F1-Score: {f1:.4f}")

### Comparing the Models

ROC curves

In [ ]:
# calculate points for the XGBoost Curve
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xg_test_probs)
xgb_auc = roc_auc_score(y_test, xg_test_probs)

# calculate points for the Random Forest Curve
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_test_probs)
rf_auc = roc_auc_score(y_test, rf_test_probs)

plt.figure(figsize=(6, 4))
# plot xgboost
plt.plot(xgb_fpr, xgb_tpr, color='lightsteelblue', lw=2, 
         label=f'XGBoost (AUC = {xgb_auc:.3f})')
# plot random forest
plt.plot(rf_fpr, rf_tpr, color='mediumpurple', lw=2, 
         label=f'Random Forest (AUC = {rf_auc:.3f})')
# plot the "Random Guess" diagonal line
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.500)')

# format plot
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity/Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve Comparison')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

Confusion Matrices

In [ ]:
xg_cm = confusion_matrix(y_test, xg_test_preds)
plt.figure(figsize=(6,4))
sns.heatmap(xg_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy', 'Diabetic'], yticklabels=['Healthy', 'Diabetic'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('XGBoost Confusion Matrix')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, rf_test_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=['Healthy', 'Diabetic'], yticklabels=['Healthy', 'Diabetic'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Random Forest Confusion Matrix')
plt.show()

Feature Importance

In [ ]:
xgboost_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xg_model.feature_importances_
})

xgboost_importances = xgboost_importances.sort_values('Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(6, 4))
sns.barplot(data=xgboost_importances.head(5), x='Importance', y='Feature', palette='Blues_r')
plt.title('XGBoost Top 5 Features')
plt.xlabel('Importance Score (Contribution)')
plt.ylabel('')
plt.show()

In [ ]:
rf_importances = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
rf_importances = rf_importances.sort_values('Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(6, 4))
sns.barplot(data=rf_importances.head(5), x='Importance', y='Feature', palette='Purples_r')
plt.title('Random Forest Top 5 Features')
plt.xlabel('Importance Score (Contribution)')
plt.ylabel('')
plt.show()

In [ ]:
# Create a figure and a set of subplots (1 row, 2 columns)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4)) #

# plot xgboost feature importance
xgboost_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xg_model.feature_importances_
})

xgboost_importances = xgboost_importances.sort_values('Importance', ascending=False).reset_index(drop=True)

sns.barplot(data=xgboost_importances.head(5), x='Importance', y='Feature', palette='Blues_r', ax=axes[0])
axes[0].set_title('XGBoost Top 5 Features')
axes[0].set_xlabel('Importance Score (Contribution)')
axes[0].set_ylabel('')

# plot rf feature importance
rf_importances = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
rf_importances = rf_importances.sort_values('Importance', ascending=False).reset_index(drop=True)

sns.barplot(data=rf_importances.head(5), x='Importance', y='Feature', palette='Purples_r', ax=axes[1])
axes[1].set_title('Random Forest Top 5 Features')
axes[1].set_xlabel('Importance Score (Contribution)')
axes[1].set_ylabel('')

plt.tight_layout() 
plt.show()

In [ ]:
# for row in rf_importances.itertuples():
    # print(f"{row.Feature}: {row.Importance:.6f}")

Precision-Recall Curves

In [ ]:
xg_precision, xg_recall, xg_thresholds = precision_recall_curve(y_test, xg_test_probs)
rf_precision, rf_recall, rf_thresholds = precision_recall_curve(y_test, rf_test_probs)
xg_pr_auc = auc(xg_recall, xg_precision)
rf_pr_auc = auc(rf_recall, rf_precision)

plt.figure(figsize=(6, 4))
plt.plot(xg_recall, xg_precision, color='lightsteelblue', lw=2, label=f'XGBoost PR Curve (AUC = {xg_pr_auc:.2f})')
plt.plot(rf_recall, rf_precision, color='mediumpurple', lw=2, label=f'Random Forest PR Curve (AUC = {rf_pr_auc:.2f})')

plt.fill_between(xg_recall, xg_precision, alpha=0.2, color='lightsteelblue')
plt.fill_between(rf_recall, rf_precision, alpha=0.2, color='mediumpurple')

# add a baseline for a random model (ratio of positive cases)
baseline_value = (y_test.sum() / len(y_test)).item()

plt.axhline(y=baseline_value, color='black', linestyle='--', label=f'Baseline ({baseline_value:.2f})')

plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Create a figure and a set of subplots (1 row, 2 columns)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4)) #

# plot ROC curves
# plot xgboost
axes[0].plot(xgb_fpr, xgb_tpr, color='lightsteelblue', lw=2, 
         label=f'XGBoost (AUC = {xgb_auc:.3f})')
# plot random forest
axes[0].plot(rf_fpr, rf_tpr, color='mediumpurple', lw=2, 
         label=f'Random Forest (AUC = {rf_auc:.3f})')
# plot the "Random Guess" diagonal line
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.500)')

# format plot
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity/Recall)')
axes[0].set_title('Receiver Operating Characteristic (ROC) Curve Comparison')
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# plot PR curves
axes[1].plot(xg_recall, xg_precision, color='lightsteelblue', lw=2, label=f'XGBoost PR Curve (AUC = {xg_pr_auc:.2f})')
axes[1].plot(rf_recall, rf_precision, color='mediumpurple', lw=2, label=f'Random Forest PR Curve (AUC = {rf_pr_auc:.2f})')
axes[1].fill_between(xg_recall, xg_precision, alpha=0.2, color='lightsteelblue')
axes[1].fill_between(rf_recall, rf_precision, alpha=0.2, color='mediumpurple')

baseline_value = (y_test.sum() / len(y_test)).item()

axes[1].axhline(y=baseline_value, color='black', linestyle='--', label=f'Baseline ({baseline_value:.2f})')

axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve Comparison')
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)


plt.tight_layout() 
plt.show()

Calibration Curve

In [ ]:
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]
rf_prob_true, rf_prob_pred = calibration_curve(y_test, rf_test_probs, n_bins=10)

xg_test_probs = xg_model.predict_proba(X_test)[:, 1]
xg_prob_true, xg_prob_pred = calibration_curve(y_test, xg_test_probs, n_bins=10)

plt.figure(figsize=(6, 4))
plt.plot(rf_prob_pred, rf_prob_true, marker='s', label='Random Forest', color='mediumpurple')
plt.plot(xg_prob_pred, xg_prob_true, marker='o', label='XGBoost', color='lightsteelblue')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly Calibrated', color='black')

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives (Actual)')
plt.title('Calibration Curve (Reliability Diagram)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

calculate 95% CI with bootstrapping method

In [ ]:

def get_full_metrics_ci(y_true, y_probs, threshold, n_bootstraps=1000):
    # Initialize lists for all metrics
    metrics = {
        'roc_auc': [], 'pr_auc': [], 'brier': [],
        'sens': [], 'spec': [], 'prec': [], 'f1': []
    }
    
    rng = np.random.RandomState(42)
    y_true = np.array(y_true).flatten()
    y_probs = np.array(y_probs).flatten()

    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_probs), len(y_probs))
        
        y_true_boot = y_true[indices]
        y_probs_boot = y_probs[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue

        # 1. Ranking Metrics (Probabilities)
        metrics['roc_auc'].append(roc_auc_score(y_true_boot, y_probs_boot))
        metrics['pr_auc'].append(average_precision_score(y_true_boot, y_probs_boot))
        metrics['brier'].append(brier_score_loss(y_true_boot, y_probs_boot))

        # 2. Classification Metrics (Binary predictions based on threshold)
        y_pred_boot = (y_probs_boot >= threshold).astype(int)

        # F1 score
        metrics['f1'].append(f1_score(y_true_boot, y_pred_boot, zero_division=0))
        
        # Sensitivity (Recall)
        metrics['sens'].append(recall_score(y_true_boot, y_pred_boot))
        
        # Precision
        metrics['prec'].append(precision_score(y_true_boot, y_pred_boot, zero_division=0))
        
        # Specificity (Requires confusion matrix: TN / (TN + FP))
        tn, fp, fn, tp = confusion_matrix(y_true_boot, y_pred_boot).ravel()
        metrics['spec'].append(tn / (tn + fp))
    
    # Calculate CIs for every list in the dictionary
    results = {}
    for metric_name, values in metrics.items():
        low, high = np.percentile(values, [2.5, 97.5])
        results[f'{metric_name}_ci'] = (low, high)
    
    return results

# --- HOW TO RUN IT FOR YOUR MODELS ---
# xgb_ci = get_full_metrics_ci(y_test, xg_test_probs, threshold=0.6357)
# rf_ci = get_full_metrics_ci(y_test, rf_test_probs, threshold=0.6201)

In [ ]:
xgb_results = get_full_metrics_ci(y_test, xg_test_probs, threshold=0.6357)
print(f"XGBoost ROC 95% CI: {xgb_results['roc_auc_ci']}")
print(f"XGBoost PR 95% CI: {xgb_results['pr_auc_ci']}")
print(f"XGBoost Brier 95% CI: {xgb_results['brier_ci']}")
print("")
print(f"XGBoost Sensitivity 95% CI: {xgb_results['sens_ci']}")
print(f"XGBoost Specificity 95% CI: {xgb_results['spec_ci']}")
print(f"XGBoost Precision 95% CI: {xgb_results['prec_ci']}")
print(f"XGBoost F1-score 95% CI: {xgb_results['f1_ci']}")

print("")
rf_results = get_full_metrics_ci(y_test, rf_test_probs, threshold=0.6201)
print(f"Random Forest ROC 95% CI: {rf_results['roc_auc_ci']}")
print(f"Random Forest PR 95% CI: {rf_results['pr_auc_ci']}")
print(f"Random Forest Brier 95% CI: {rf_results['brier_ci']}")
print("")
print(f"Random Forest Sensitivity 95% CI: {rf_results['sens_ci']}")
print(f"Random Forest Specificity 95% CI: {rf_results['spec_ci']}")
print(f"Random Forest Precision 95% CI: {rf_results['prec_ci']}")
print(f"Random Forest F1 95% CI: {rf_results['f1_ci']}")

print("")

In [ ]:
def calculate_bootstrap_p_value(metric_a, metric_b):
    """
    Calculates the p-value for the difference between two models.
    metric_a: list/array of bootstrap results for Model A (e.g., XGBoost)
    metric_b: list/array of bootstrap results for Model B (e.g., Random Forest)
    """
    # Calculate the difference for every single iteration
    # This accounts for the 'correlation' of the errors
    deltas = np.array(metric_a) - np.array(metric_b)
    
    # Proportion of times the difference is on the 'wrong' side of zero
    # We multiply by 2 for a two-tailed test
    p_value = 2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0))
    
    return p_value

# Example usage for your Brier Scores:
# p_val_brier = calculate_bootstrap_p_value(xgboost_brier_list, rf_brier_list)
# print(f"P-value for Brier Score difference: {p_val_brier:.4f}")

In [ ]:
# calculate p vals
p_val_roc = calculate_bootstrap_p_value(xgb_results['roc_auc_ci'], rf_results['roc_auc_ci'])
print(f"P-value for ROC Score difference: {p_val_roc:.4f}")
p_val_pr = calculate_bootstrap_p_value(xgb_results['pr_auc_ci'], rf_results['pr_auc_ci'])
print(f"P-value for PR Score difference: {p_val_pr:.4f}")
p_val_brier = calculate_bootstrap_p_value(xgb_results['brier_ci'], rf_results['brier_ci'])
print(f"P-value for Brier Score difference: {p_val_brier:.4f}")
print("")

p_val_sens = calculate_bootstrap_p_value(xgb_results['sens_ci'], rf_results['sens_ci'])
print(f"P-value for sensitivity difference: {p_val_sens:.4f}")
p_val_spec = calculate_bootstrap_p_value(xgb_results['spec_ci'], rf_results['spec_ci'])
print(f"P-value for specificity difference: {p_val_spec:.4f}")
p_val_prec = calculate_bootstrap_p_value(xgb_results['prec_ci'], rf_results['prec_ci'])
print(f"P-value for precision difference: {p_val_prec:.4f}")
p_val_f1 = calculate_bootstrap_p_value(xgb_results['f1_ci'], rf_results['f1_ci'])
print(f"P-value for f1 difference: {p_val_f1:.4e}")

In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, 
                             f1_score, recall_score, precision_score, confusion_matrix)

def get_paired_bootstrap_results(y_true, probs_a, probs_b, threshold_a, threshold_b, n_bootstraps=1000):
    # Initialize dictionary to store distributions
    metrics = ['roc_auc', 'pr_auc', 'brier', 'sens', 'spec', 'prec', 'f1']
    scores = {
        'model_a': {m: [] for m in metrics},
        'model_b': {m: [] for m in metrics}
    }
    
    rng = np.random.RandomState(42)
    y_true = np.array(y_true).flatten()
    probs_a = np.array(probs_a).flatten()
    probs_b = np.array(probs_b).flatten()

    for i in range(n_bootstraps):
        # Sampling indices ONCE per iteration ensures a PAIRED comparison
        indices = rng.randint(0, len(y_true), len(y_true))
        
        y_true_boot = y_true[indices]
        # Ensure bootstrap sample contains both classes
        if len(np.unique(y_true_boot)) < 2:
            continue

        # --- MODEL A (XGBoost) ---
        p_a = probs_a[indices]
        y_pred_a = (p_a >= threshold_a).astype(int)
        tn_a, fp_a, fn_a, tp_a = confusion_matrix(y_true_boot, y_pred_a).ravel()
        
        scores['model_a']['roc_auc'].append(roc_auc_score(y_true_boot, p_a))
        scores['model_a']['pr_auc'].append(average_precision_score(y_true_boot, p_a))
        scores['model_a']['brier'].append(brier_score_loss(y_true_boot, p_a))
        scores['model_a']['f1'].append(f1_score(y_true_boot, y_pred_a, zero_division=0))
        scores['model_a']['sens'].append(recall_score(y_true_boot, y_pred_a))
        scores['model_a']['prec'].append(precision_score(y_true_boot, y_pred_a, zero_division=0))
        scores['model_a']['spec'].append(tn_a / (tn_a + fp_a))

        # --- MODEL B (Random Forest) ---
        p_b = probs_b[indices]
        y_pred_b = (p_b >= threshold_b).astype(int)
        tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_true_boot, y_pred_b).ravel()
        
        scores['model_b']['roc_auc'].append(roc_auc_score(y_true_boot, p_b))
        scores['model_b']['pr_auc'].append(average_precision_score(y_true_boot, p_b))
        scores['model_b']['brier'].append(brier_score_loss(y_true_boot, p_b))
        scores['model_b']['f1'].append(f1_score(y_true_boot, y_pred_b, zero_division=0))
        scores['model_b']['sens'].append(recall_score(y_true_boot, y_pred_b))
        scores['model_b']['prec'].append(precision_score(y_true_boot, y_pred_b, zero_division=0))
        scores['model_b']['spec'].append(tn_b / (tn_b + fp_b))
    
    return scores

In [ ]:
# Run the simulation
raw_data = get_paired_bootstrap_results(y_test, xg_test_probs, rf_test_probs, 0.6357, 0.6201)

print(f"{'Metric':<10} | {'XGBoost (Mean [95% CI])':<25} | {'RF (Mean [95% CI])':<25} | {'p-value'}")
print("-" * 85)

for m in ['roc_auc', 'pr_auc', 'brier', 'f1', 'sens', 'spec', 'prec']:
    a_vals = np.array(raw_data['model_a'][m])
    b_vals = np.array(raw_data['model_b'][m])
    
    # calculate stats for table
    a_mu, a_low, a_high = np.mean(a_vals), np.percentile(a_vals, 2.5), np.percentile(a_vals, 97.5)
    b_mu, b_low, b_high = np.mean(b_vals), np.percentile(b_vals, 2.5), np.percentile(b_vals, 97.5)
    
    # calculate p-value (two-tailed bootstrap difference)
    deltas = a_vals - b_vals
    p_val = 2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0))
    
    # format table
    a_str = f"{a_mu:.4f} [{a_low:.4f}-{a_high:.4f}]"
    b_str = f"{b_mu:.4f} [{b_low:.4f}-{b_high:.4f}]"
    
    print(f"{m:<10} | {a_str:<25} | {b_str:<25} | {p_val:.4e}")

In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, 
                             f1_score, recall_score, precision_score, confusion_matrix)

def get_full_paired_analysis(y_true, probs_a, probs_b, threshold_a, threshold_b, n_bootstraps=1000):
    metrics_list = ['roc_auc', 'pr_auc', 'brier', 'sens', 'spec', 'prec', 'f1']
    
    # Storage for raw bootstrap distributions
    raw_scores = {
        'model_a': {m: [] for m in metrics_list},
        'model_b': {m: [] for m in metrics_list},
        'deltas':  {m: [] for m in metrics_list}
    }
    
    rng = np.random.RandomState(42)
    y_true = np.array(y_true).flatten()
    probs_a = np.array(probs_a).flatten()
    probs_b = np.array(probs_b).flatten()

    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_true), len(y_true))
        y_true_boot = y_true[indices]
        
        if len(np.unique(y_true_boot)) < 2: continue

        # --- Data extraction for this fold ---
        p_a, p_b = probs_a[indices], probs_b[indices]
        y_a, y_b = (p_a >= threshold_a).astype(int), (p_b >= threshold_b).astype(int)
        
        # --- Metric Calculation Helpers ---
        def get_metrics(y_t, p_scores, y_preds):
            tn, fp, fn, tp = confusion_matrix(y_t, y_preds).ravel()
            return {
                'roc_auc': roc_auc_score(y_t, p_scores),
                'pr_auc':  average_precision_score(y_t, p_scores),
                'brier':   brier_score_loss(y_t, p_scores),
                'sens':    recall_score(y_t, y_preds),
                'spec':    tn / (tn + fp),
                'prec':    precision_score(y_t, y_preds, zero_division=0),
                'f1':      f1_score(y_t, y_preds, zero_division=0)
            }

        res_a = get_metrics(y_true_boot, p_a, y_a)
        res_b = get_metrics(y_true_boot, p_b, y_b)

        # --- Store results and calculate paired Delta ---
        for m in metrics_list:
            raw_scores['model_a'][m].append(res_a[m])
            raw_scores['model_b'][m].append(res_b[m])
            raw_scores['deltas'][m].append(res_a[m] - res_b[m]) # Paired Difference
    
    # --- Final Summary Statistics ---
    summary = {}
    for m in metrics_list:
        # P-value calculation (two-tailed)
        d = np.array(raw_scores['deltas'][m])
        p_val = 2 * min(np.mean(d <= 0), np.mean(d >= 0))
        
        # CI of the Difference
        d_mean = np.mean(d)
        d_low, d_high = np.percentile(d, [2.5, 97.5])
        
        summary[m] = {
            'p_value': p_val,
            'delta_mean': d_mean,
            'delta_ci': (d_low, d_high),
            'model_a_ci': (np.percentile(raw_scores['model_a'][m], 2.5), np.percentile(raw_scores['model_a'][m], 97.5)),
            'model_b_ci': (np.percentile(raw_scores['model_b'][m], 2.5), np.percentile(raw_scores['model_b'][m], 97.5))
        }
        
    return summary

In [ ]:
# 1. Define your optimal thresholds (from your previous work)
threshold_xgb = 0.6357
threshold_rf = 0.6201

# 2. Call the function
# This will run 1,000 simulations where both models are tested on the same data slices
analysis_results = get_full_paired_analysis(
    y_true=y_test, 
    probs_a=xg_test_probs,  # XGBoost probabilities
    probs_b=rf_test_probs,  # Random Forest probabilities
    threshold_a=threshold_xgb, 
    threshold_b=threshold_rf, 
    n_bootstraps=1000
)

In [ ]:
# Loop through the metrics to see the "Delta CI" and "P-value"
for metric, stats in analysis_results.items():
    print(f"--- {metric.upper()} ---")
    print(f"Paired Difference Mean: {stats['delta_mean']:.4f}")
    print(f"95% CI of Difference:  [{stats['delta_ci'][0]:.4f}, {stats['delta_ci'][1]:.4f}]")
    print(f"P-value:               {stats['p_value']:.4f}")
    
    # Check if significant (if CI does not cross 0)
    is_sig = "YES" if (stats['delta_ci'][0] > 0 or stats['delta_ci'][1] < 0) else "NO"
    print(f"Statistically Significant? {is_sig}")
    print("\n")

In [ ]:
import pauc # or from MLstatkit import Delong_test

# 2. If using pAUC:
roc_xgb = pauc.ROC(np.squeeze(y_test.values.T), xg_test_probs, name="XGBoost")
roc_rf = pauc.ROC(np.squeeze(y_test.values.T), rf_test_probs, name="Random Forest")

# 3. Perform the comparison
comparison = pauc.compare(roc_xgb, roc_rf, method="delong")

print(comparison)

In [ ]:

# create the explainer (TreeExplainer is the fastest and most accurate for tree models)
xg_explainer = shap.TreeExplainer(xg_model)

# calculate SHAP values
xg_shap_values = xg_explainer.shap_values(X_test)

# summary plot shows the magnitude and direction of feature impacts
shap.summary_plot(xg_shap_values, X_test, max_display = 10)

In [ ]:
shap.dependence_plot("HighBP", xg_shap_values, X_test)

In [ ]:
shap.dependence_plot("BMI", xg_shap_values, X_test)

In [ ]:
# this takes ~15 minutes

# create the explainer (TreeExplainer is the fastest and most accurate for tree models)
rf_explainer = shap.TreeExplainer(rf_model)

# calculate SHAP values
rf_shap_values = rf_explainer.shap_values(X_test)
rf_shap_pos = rf_shap_values[:, :, 1]

# summary plot shows the magnitude and direction of feature impacts
shap.summary_plot(rf_shap_pos, X_test, max_display=10)

In [ ]:
# random forest shap plot will all features for supplemental
plt.figure()
shap.summary_plot(rf_shap_pos, X_test,show=False,max_display=10)
plt.title("Random Forest: SHAP Summary Plot of Top 10 Features")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot Feature A on the first axis
shap.dependence_plot("BMI", xg_shap_values, X_test, ax=ax1, show=False)
ax1.set_title("XGBoost: BMI Dependence Plot")
# Plot Feature B on the second axis
# By default, SHAP picks the best interaction feature to color the dots
shap.dependence_plot("BMI", rf_shap_pos, X_test, ax=ax2, show=False)
ax2.set_title("Random Forest: BMI Dependence Plot")

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot Feature A on the first axis
shap.dependence_plot("Age", xg_shap_values, X_test, ax=ax1, show=False)
ax1.set_title("XGBoost: Age Dependence Plot")
# Plot Feature B on the second axis
# By default, SHAP picks the best interaction feature to color the dots
shap.dependence_plot("Age", rf_shap_pos, X_test, ax=ax2, show=False)
ax2.set_title("Random Forest: Age Dependence Plot")

plt.tight_layout()
plt.show()

In [ ]:
# xgb shap plot will all features for supplemental
plt.figure()
shap.summary_plot(xg_shap_values, X_test,show=False,max_display=10)
plt.title("XGBoost: SHAP Summary Plot of Top 10 Features")
plt.show()

#### Comparing AUC - Delong's Test

[this code is from Yandex Data School ROC comparison code]

In [ ]:
import scipy

def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N)

    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5*(i+j-1)
        i = j

    T2 = np.empty(N)
    T2[J] = T + 1
    return T2


def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m

    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]

    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m])
    ty = np.empty([k, n])
    tz = np.empty([k, m+n])

    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1)/m/n - (m+1.0)/2.0/n

    v01 = (tz[:, :m] - tx)/n
    v10 = 1.0 - (tz[:, m:] - ty)/m

    sx = np.cov(v01)
    sy = np.cov(v10)

    delongcov = sx/m + sy/n
    return aucs, delongcov


def delong_roc_test(y_true, pred1, pred2):
    order = np.argsort(-y_true)
    label_1_count = int(np.sum(y_true))

    predictions = np.vstack((pred1, pred2))[:, order]

    aucs, delongcov = fast_delong(predictions, label_1_count)

    diff = aucs[0] - aucs[1]
    var = delongcov[0,0] + delongcov[1,1] - 2*delongcov[0,1]
    z = np.abs(diff) / np.sqrt(var)

    p = 2 * (1 - scipy.stats.norm.cdf(z))
    return p

In [ ]:
flat_array =  y_test.to_numpy().ravel()
p_value = delong_roc_test(flat_array, rf_test_probs, xg_test_probs)
print("DeLong p-value:", p_value)

#### Error Analysis

In [ ]:
# fixing age column to have better numbers
age_label = ['18-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70-74','75-79','80 or older']
age_mapping = {i: label for i, label in enumerate(age_label, start=1)}
X_test['Age_Group'] = X_test['Age'].map(age_mapping)
X_test.head(2)

In [ ]:
xg_results = pd.DataFrame({
    "true": flat_array,
    "pred": xg_test_preds,
    "Age": X_test["Age_Group"],
    "HighBP": X_test["HighBP"],
    "BMI": X_test["BMI"],
    "HighChol": X_test["HighChol"]
})
xg_results["error"] = xg_results["true"] != xg_results["pred"]
rf_results = pd.DataFrame({
    "true": flat_array,
    "pred": rf_test_preds,
    "Age": X_test["Age_Group"],
    "HighBP": X_test["HighBP"],
    "BMI": X_test["BMI"],
    "HighChol": X_test["HighChol"]
})
rf_results["error"] = rf_results["true"] != rf_results["pred"]

Blood Pressure Errors

In [ ]:
print("XG Boost BP")
print(xg_results.groupby("HighBP")["error"].mean())
print("random forest BP")
print(rf_results.groupby("HighBP")["error"].mean())

Age Errors

In [ ]:
print("XG Boost Age")
print(xg_results.groupby("Age")["error"].mean())
print("Random ForestAge")
print(rf_results.groupby("Age")["error"].mean())

Cholesterol Errors

In [ ]:
# include size of each group to make sure we aren't just seeing noise from small groups
print("XG Boost Cholesterol")
print(xg_results.groupby("HighChol")["error"].mean())
print(xg_results.groupby("HighChol").size())
print("random forest Cholesterol")
print(rf_results.groupby("HighChol")["error"].mean())
print(rf_results.groupby("HighChol").size())

BMI Errors

In [ ]:
xg_results["BMI_bin"] = pd.cut(xg_results["BMI"], bins=5)
xg_results.groupby("BMI_bin")["error"].mean()

In [ ]:
rf_results["BMI_bin"] = pd.cut(rf_results["BMI"], bins=5)
rf_results.groupby("BMI_bin")["error"].mean()